In [1]:
import pandas as pd 
import numpy as np 
# import matplotlib.pyplot as plt
from pyfaidx import Fasta
# import regex
from sklearn.model_selection import train_test_split
import os
import random

%matplotlib inline
pd.options.mode.chained_assignment = None 
np.random.seed = 42

Creat data set with subsampling negative instances by shuffling positives at single bps level.

In [2]:
def add_chr_prefix(chromosome):
    return f'chr{chromosome}'

In [3]:
#Randomly extended sequnce Window=500
ex_yeast_oridb = pd.read_csv("/p/project/hai_dnaori/piroozeh1/yeast-origins/data/window_dataset_indices_oridb.csv")

ex_yeast_oridb = ex_yeast_oridb[ex_yeast_oridb['label'] == 1]
ex_yeast_oridb


,chr,start,end,label
3,1,30781,31280,1
5,1,70170,70669,1
7,1,124264,124763,1
8,1,159903,160402,1
9,1,175911,176410,1
...,...,...,...,...
641,16,776744,777243,1
643,16,818960,819459,1
644,16,842595,843094,1
645,16,880682,881181,1


In [2]:
def seq2kmer(seq, k):
        """
        Convert original sequence to kmers
        
        Arguments:
        seq -- str, original sequence.
        k -- int, kmer of length k specified.
        
        Returns:
        kmers -- str, kmers separated by space

        """
        kmer = [seq[x:x+k] for x in range(len(seq)+1-k)]
        kmers = " ".join(kmer)
        return kmers

    # add kmers with k = 4
# full_dataset["sequence"] = full_dataset.apply(lambda x: seq2kmer(x.seq, 4), axis=1)
# full_dataset

In [5]:
def shuffle_dna(sequence):
    seq_list = list(sequence)
    
    # Shuffle the list
    random.shuffle(seq_list)
    
    # Join the shuffled list back into a string
    shuffled_seq = ''.join(seq_list)
    
    return shuffled_seq
    # return ''.join(np.random.permutation(sequence))

In [6]:
seq3= "ATTGCT"
sh=shuffle_dna(seq3)
sh

'TAGTCT'

In [7]:
sequence_data = Fasta(f"/p/project/hai_dnaori/sgd_data/S288C_reference_sequence_R64-3-1_20210421.fsa")
ex_yeast_oridb["seq"] = ex_yeast_oridb.apply(lambda x: sequence_data[f"chr{x.chr}"][x.start-1: x.end].seq, axis=1)
ex_yeast_oridb


,chr,start,end,label,seq
3,1,30781,31280,1,ATATTTTAATGTTAAGATGAAATTTAAGTGAGCTGGTAATATCAAG...
5,1,70170,70669,1,AAGTGAGAAGAAAAAAAAAGGAAAAAAAGGAATTGTCCTAATGAGC...
7,1,124264,124763,1,ACATAAATAACGACAAATGGTTGGTTATTTGAAGGATTAATGATCA...
8,1,159903,160402,1,ATGCATTCAGCGGGAAAGTAGTTGTTTATCACTAGACATATAATTA...
9,1,175911,176410,1,AACGATATTCGATAATGCGCCAAGCCTTTATAAGGAACTCAAAATA...
...,...,...,...,...,...
641,16,776744,777243,1,TACCAAAAATTCTCTCTGAGGATATAGGAATCTACAAAATGAATCT...
643,16,818960,819459,1,CCGAAAATTTAAGCGAAATTAAGAATAAAGAAGAAAAGAAAGAATT...
644,16,842595,843094,1,TGGCTGAGAGGAATTCTAAGATTTCTAATGTGGAGAGGTATACTCT...
645,16,880682,881181,1,GGATTGTCAAGACACTCCGGTATTACTCGAGCCCGTAATACAACAG...


In [8]:
posetive_samples=ex_yeast_oridb[['chr', 'seq','label','start','end' ]]
negative_samples= posetive_samples.copy()
negative_samples['label'] = 0

print(negative_samples)


# print(negative_samples)

     chr                                                seq  label   start  \
3      1  ATATTTTAATGTTAAGATGAAATTTAAGTGAGCTGGTAATATCAAG...      0   30781   
5      1  AAGTGAGAAGAAAAAAAAAGGAAAAAAAGGAATTGTCCTAATGAGC...      0   70170   
7      1  ACATAAATAACGACAAATGGTTGGTTATTTGAAGGATTAATGATCA...      0  124264   
8      1  ATGCATTCAGCGGGAAAGTAGTTGTTTATCACTAGACATATAATTA...      0  159903   
9      1  AACGATATTCGATAATGCGCCAAGCCTTTATAAGGAACTCAAAATA...      0  175911   
..   ...                                                ...    ...     ...   
641   16  TACCAAAAATTCTCTCTGAGGATATAGGAATCTACAAAATGAATCT...      0  776744   
643   16  CCGAAAATTTAAGCGAAATTAAGAATAAAGAAGAAAAGAAAGAATT...      0  818960   
644   16  TGGCTGAGAGGAATTCTAAGATTTCTAATGTGGAGAGGTATACTCT...      0  842595   
645   16  GGATTGTCAAGACACTCCGGTATTACTCGAGCCCGTAATACAACAG...      0  880682   
649   16  TATAAATGGGTACCAAGGCATTAAATATAGATCTGGTTTACTAATC...      0  932894   

        end  
3     31280  
5     70669  
7    124763  
8    16

In [12]:
# for index ,sample in negative_samples.iterrows():
# shuffled= shuffle_dna(sample['seq'])
#     negative_samples.at[index,'seq']=shuffle_dna(sample['seq'])
negative_samples['seq'] = negative_samples['seq'].apply(shuffle_dna)
print(negative_samples)
print(posetive_samples)    

     chr                                                seq  label   start  \
3      1  TAGGGGCGATATAACATCAATTGAGTCCGTAGACATACTCACTCTA...      0   30781   
5      1  AGAATCAATGATTAGAAATTCTCACAAATAGCATGTTAGGATCAAT...      0   70170   
7      1  TATGACAAATTAATAATCAAAACCTATAACGATACATATGAGTCAT...      0  124264   
8      1  TTCTAGAAAGATAGCAACGGGATCATTTCCATTTTACAGACTAGTC...      0  159903   
9      1  TCTGATGTACTGCAAAAAAATTTAACAAGAACACTCAGAACTAAGC...      0  175911   
..   ...                                                ...    ...     ...   
641   16  ATTAACAAAGATTGTATTAGCTTGCGAGAACTTTATTTGTTATCTA...      0  776744   
643   16  CTGTATATCTTAAAACATAGATACCTTCCAATATATTATAGGGTTC...      0  818960   
644   16  TAAGTAGTTTGAAAGTATATTTTAGAGACCTATATATTAATGCAAT...      0  842595   
645   16  TTAATGTATACTTGTGTATAAACATTTAAATGACTTGACGCTATAA...      0  880682   
649   16  GAGAATTATGAGAAATAAAGTTGTAACACAGCAATTGATTATAATA...      0  932894   

        end  
3     31280  
5     70669  
7    124763  
8    16

In [15]:
filtered_rows = negative_samples[(negative_samples['chr'] == 1) & (negative_samples['start'] == 70170)]
filtered_rows


,chr,seq,label,start,end
5,1,AGAATCAATGATTAGAAATTCTCACAAATAGCATGTTAGGATCAAT...,0,70170,70669


In [16]:
import pandas as pd
full_dataset = pd.concat([posetive_samples, negative_samples], ignore_index=True)
# full_dataset = posetive_samples.append(pd.DataFrame(negative_samples,columns=posetive_samples.columns, ignore_index=True))

full_dataset.sort_values(['chr', 'start'], inplace=True)

# full_dataset.to_csv("/p/project/hai_dnaori/piroozeh1/yeast-origins/data/OriDB_Shuffled/new_shuffeld/oridb_with_shuffled_negative2.csv", index=False)
full_dataset





,chr,seq,label,start,end
0,1,ATATTTTAATGTTAAGATGAAATTTAAGTGAGCTGGTAATATCAAG...,1,30781,31280
325,1,TAGGGGCGATATAACATCAATTGAGTCCGTAGACATACTCACTCTA...,0,30781,31280
1,1,AAGTGAGAAGAAAAAAAAAGGAAAAAAAGGAATTGTCCTAATGAGC...,1,70170,70669
326,1,AGAATCAATGATTAGAAATTCTCACAAATAGCATGTTAGGATCAAT...,0,70170,70669
2,1,ACATAAATAACGACAAATGGTTGGTTATTTGAAGGATTAATGATCA...,1,124264,124763
...,...,...,...,...,...
647,16,TAAGTAGTTTGAAAGTATATTTTAGAGACCTATATATTAATGCAAT...,0,842595,843094
323,16,GGATTGTCAAGACACTCCGGTATTACTCGAGCCCGTAATACAACAG...,1,880682,881181
648,16,TTAATGTATACTTGTGTATAAACATTTAAATGACTTGACGCTATAA...,0,880682,881181
324,16,TATAAATGGGTACCAAGGCATTAAATATAGATCTGGTTTACTAATC...,1,932894,933393


In [9]:
#fetch and prepare train dataset represented by range

data_path_DNABER="/p/project/hai_dnaori/piroozeh1/yeast-origins/data/OriDB_Shuffled/new_shuffled/DNABERT"
data_path_DNABER2="/p/project/hai_dnaori/piroozeh1/yeast-origins/data/OriDB_Shuffled/new_shuffled/DNABERT2"

full_dataset = pd.read_csv("/p/project/hai_dnaori/piroozeh1/yeast-origins/data/OriDB_Shuffled/new_shuffled/oridb_with_shuffled_negative.csv")
                            
full_dataset["kmer"] = full_dataset.apply(lambda x: seq2kmer(x.seq, 3), axis=1)
full_dataset


# train test split
X_train, X_test, y_train, y_test = train_test_split(full_dataset[["chr", "seq", "kmer", "start", "end"]], full_dataset.label, test_size=0.2, random_state=100, stratify=full_dataset.label)

# train valid split
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.125, random_state=100, stratify=y_train)
# train_new = pd.DataFrame({"sequence": X_train.seq, "label": y_train.values})
# valid_new = pd.DataFrame({"sequence": X_valid.seq, "label": y_valid.values})
# test_new = pd.DataFrame({"sequence": X_test.seq, "label": y_test.values})

train_new_kmer = pd.DataFrame({"sequence": X_train.kmer, "label": y_train.values})
valid_new_kmer = pd.DataFrame({"sequence": X_valid.kmer, "label": y_valid.values})
test_new_kmer = pd.DataFrame({"sequence": X_test.kmer, "label": y_test.values})

# save final kmer files to tsv
train_new_kmer.to_csv(os.path.join(data_path_DNABER, "train_dev_test100/3/train.tsv"), sep="\t", index=False)
valid_new_kmer.to_csv(os.path.join(data_path_DNABER,"train_dev_test100/3/dev.tsv"), sep="\t", index=False)
test_new_kmer.to_csv(os.path.join(data_path_DNABER,"train_dev_test100/3/test.tsv"), sep="\t", index=False)

# # save final files to tsv
# train_new.to_csv(os.path.join(data_path_DNABER,"train_dev_test30/train.tsv"), sep="\t", index=False)
# valid_new.to_csv(os.path.join(data_path_DNABER,"train_dev_test30/dev.tsv"), sep="\t", index=False)
# test_new.to_csv(os.path.join(data_path_DNABER,"train_dev_test30/test.tsv"), sep="\t", index=False)

# #DNABERT2
# train_new.to_csv(os.path.join(data_path_DNABER2,"train_dev_test30/train.csv"), index=False)
# valid_new.to_csv(os.path.join(data_path_DNABER2,"train_dev_test30/dev.csv"), index=False)
# test_new.to_csv(os.path.join(data_path_DNABER2,"train_dev_test30/test.csv"), index=False)




# train_window = pd.DataFrame({"chr": X_train.chr, "start": X_train.start, "end": X_train.end,"seq": X_train.seq, "label": y_train.values})
# valid_window = pd.DataFrame({"chr": X_valid.chr, "start": X_valid.start, "end": X_valid.end,"seq": X_valid.seq, "label": y_valid.values})
# test_window = pd.DataFrame({"chr": X_test.chr, "start": X_test.start, "end": X_test.end,"seq": X_test.seq, "label": y_test.values})


# # train_window.drop(columns=["seq"], inplace=True)
# train_window.to_csv(os.path.join(data_path_DNABER, "train_dev_test100/train_window.tsv"), sep="\t", index=False)
# valid_window.to_csv(os.path.join(data_path_DNABER, "train_dev_test100/dev_window.tsv"), sep="\t", index=False)
# test_window.to_csv(os.path.join(data_path_DNABER, "train_dev_test100/test_window.tsv"), sep="\t", index=False)
# # 
# train_window.drop(columns=["seq"], inplace=True)
# valid_window.drop(columns=["seq"], inplace=True)
# test_window.drop(columns=["seq"], inplace=True)

# train_window.to_csv(os.path.join(data_path_DNABER, "train_dev_test100/train_window.bed"),sep='\t', header=False, index=False)
# valid_window.to_csv(os.path.join(data_path_DNABER, "train_dev_test100/dev_window.bed"),sep='\t', header=False, index=False)
# test_window.to_csv(os.path.join(data_path_DNABER, "train_dev_test100/test_window.bed"),sep='\t', header=False, index=False)


